In [1]:
from gsloc.inference.test import TestConfig, Test
from pathlib import Path
from gsloc.models import opr_graph_extention as network 
import torch
from torchvision.transforms import functional as F
from mmpr.models import MegaLoc
from gsloc.datasets import ThreeRScan

from torchvision import transforms as T
from gsloc.utils.visual import plot_metrics_from_parquet, plot_metrics_from_experiment_dir

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


2026-05-06 23:47:53.553 | WARNING  | opr.optional_deps:warn_once:115 - MinkowskiEngine is not available. sparse convolutions will be disabled. See the documentation for installation instructions


In [2]:
weights_path = Path("/mnt/external_usb_hdd/6YL/Datasets/3RScan/weights/graphs/gatv1/best_model.pth")
ckpt = torch.load(weights_path, map_location="cpu", weights_only=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

OPR_GAT_graph_encoder = network.OPR_GATGraphEncoder(
    in_dim=4,
    hidden_dim=512,
    n_layers=1,
    num_node_classes=529, 
    node_emb_dim=128,
    num_edge_classes=41,
    edge_emb_dim=128,
    proj_dim=256,
    edge_cont_dim=10,
    dropout=0.1,
    heads=4
    ).to(device)
    
# megaloc = torch.hub.load("gmberton/MegaLoc", "get_trained_model")
# image_encoder = megaloc.to(device)

graph_model1 = network.OPR_MultiModalVPRGraphEncoder(
    graph_encoder=OPR_GAT_graph_encoder,
    image_encoder=None,
    image_out_dim=8448,
    graph_out_dim=256,
    fusion_dim=8448,
    normalize=True,
    graph_fusion_scale=0.05,
    freeze_image_encoder=True,
    mode="graph")

missing, unexpected = graph_model1.load_state_dict(ckpt["model_state_dict"], strict=False)
ignored_unexpected_prefixes = ("image_encoder.", "graph_encoder.convs.")
unexpected_other = [k for k in unexpected if not k.startswith(ignored_unexpected_prefixes)]
if unexpected_other:
    raise RuntimeError(f"Unexpected checkpoint keys: {unexpected_other}")
# ``missing`` includes MegaLoc hub weights and GINE conv params; those ckpt tensors appear under ``ignored_unexpected_prefixes``.

graph_model1.to(device)
graph_model1.eval()

OPR_MultiModalVPRGraphEncoder(
  (graph_encoder): OPR_GATGraphEncoder(
    (edge_cont_ln): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (edge_lbl_ln): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (node_emb): Embedding(529, 128)
    (edge_emb): Embedding(41, 128)
    (edge_cont_mlp): Sequential(
      (0): Linear(in_features=10, out_features=512, bias=True)
      (1): ReLU(inplace=True)
      (2): Linear(in_features=512, out_features=512, bias=True)
    )
    (edge_gate): Sequential(
      (0): Linear(in_features=1024, out_features=512, bias=True)
      (1): ReLU(inplace=True)
      (2): Linear(in_features=512, out_features=512, bias=True)
      (3): Sigmoid()
    )
    (edge_label_proj): Sequential(
      (0): Linear(in_features=128, out_features=512, bias=True)
      (1): ReLU(inplace=True)
      (2): Linear(in_features=512, out_features=512, bias=True)
    )
    (edge_fuse): Sequential(
      (0): Linear(in_features=1024, out_features=512, bias=True)
      (

In [3]:
megaLoc = MegaLoc()
megaLoc.to(device)
megaLoc.eval()

Using cache found in /home/kartashov_ga/.cache/torch/hub/gmberton_MegaLoc_main
Using cache found in /home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main
/home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


MegaLoc(
  (model): MegaLocModel(
    (backbone): DINOv2(
      (model): DinoVisionTransformer(
        (patch_embed): PatchEmbed(
          (proj): Conv2d(3, 768, kernel_size=(14, 14), stride=(14, 14))
          (norm): Identity()
        )
        (blocks): ModuleList(
          (0-11): 12 x NestedTensorBlock(
            (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
            (attn): MemEffAttention(
              (qkv): Linear(in_features=768, out_features=2304, bias=True)
              (proj): Linear(in_features=768, out_features=768, bias=True)
              (proj_drop): Dropout(p=0.0, inplace=False)
            )
            (ls1): LayerScale()
            (drop_path1): Identity()
            (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
            (mlp): Mlp(
              (fc1): Linear(in_features=768, out_features=3072, bias=True)
              (act): GELU(approximate='none')
              (fc2): Linear(in_features=3072, out_features=768, 

In [4]:
dataset_path = Path("/mnt/external_usb_hdd/6YL/Datasets/3RScan")
test_dir = Path("/home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/256xMegaloc")
index_path = "/home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/256_graph_index"
rerank_index_path = "/home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/megaloc_index"
query_cache_path = "/home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/256_graph_query_cache"
rerank_query_cache_path = "/home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/megaloc_query_cache"

graph_path = "SceneGraphs_Makarov_FULL_TEST_pt"
edge_normalizer_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/weights/graphs/gatv1/edge_normalizer (1).pt"
scene_list_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/test_resplit_scans.txt"
room_json_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/3RScan.json"

bench_report_dir = test_dir / "bench_reports" / "pose-near-sim"
frames_path = test_dir / "bench_reports" / "frames.npz"

seq_filter_kwargs_list = [{
    "seq_similarity_filter_mode": "none",
},
{
    "seq_similarity_filter_mode": "pose",
    "seq_similarity_trans_tol_m": 0.5,
    "seq_similarity_rot_tol_deg": 15
},{
    "seq_similarity_filter_mode": "pose",
    "seq_similarity_trans_tol_m": 1,
    "seq_similarity_rot_tol_deg": 30
},
]
models = [graph_model1]
rerank_models = [megaLoc]

graph_path_list = [graph_path, "SceneGraphs_real_classes_pt_compact"]
similarity_kwargs_list = [
    {
        "mode": "room",
    },
    {
        "mode": "pose",
        "trans_tol_m": 3,
        "rot_tol_deg": 180
    },
    {
        "mode": "pose",
        "trans_tol_m": 2,
        "rot_tol_deg": 90
    },
]


image_transform_fn = T.Compose([
    T.ToTensor(),
    T.Lambda(lambda x: F.rotate(x, angle=-90)),  # 90° clockwise
    T.Normalize(mean=[0.44420420130352495, 0.41322746532289134, 0.3678658064565412], std=[0.24352604373543688, 0.24045797651069503, 0.24250136992133814]),
    T.Resize([322, 322], antialias=True)
])

cfg = TestConfig(
    dataset_path=dataset_path,
    test_path=test_dir,
    index_path=index_path,
    rerank_index_path=rerank_index_path,
    query_cache_path=query_cache_path,
    rerank_query_cache_path=rerank_query_cache_path,
    bench_report_path=bench_report_dir,
    graph_path=graph_path,   
    dataset_class=ThreeRScan,
    filter_kwargs={"similarity_filter_mode": "none", "similarity_trans_tol_m": 2, "similarity_rot_tol_deg": 90},
    seq_filter_kwargs={"seq_similarity_filter_mode": "none", "seq_similarity_trans_tol_m": 2, "seq_similarity_rot_tol_deg": 90},
    scene_list_path=scene_list_path,
    room_json_path=room_json_path,
    edge_normalizer_path=edge_normalizer_path,
    image_transform_fn=image_transform_fn,
    graph_feat_dim=4,
    graph_edge_attr_dim=10,
    graph_rotate=True,
    device=device,
    batch_size=16,
    num_workers=4,
    model=graph_model1,
    rerank_model=megaLoc,
    rerank_k=500,
    per_frame_k_used=25,
    final_k=25,
    seq_lengths=[1, 2, 3, 5, 7, 10, 15, 20, 25, 30, 35],
    recall_at_k=[1, 5, 10, 25],
    similarity_kwargs=similarity_kwargs_list[2],
    std_mode="global",
    scene_df_field="scene",
    pose_df_field="pose",
    frames_path=frames_path
)

In [5]:
similarity_names = ["room-sim", "pose-far-sim", "pose-near-sim"]
filter_names = ["base_seq_report", "nearfilter_seq_report", "farfilter_seq_report"]

In [6]:
# cfg.seq_filter_kwargs = seq_filter_kwargs_list[1]
# cfg.bench_report_path = cfg.test_path / filter_names[1] / similarity_names[0]
# cfg.frames_path = cfg.test_path / "frames.npz"
# test = Test(cfg)
# test.run()

In [7]:
# cfg.seq_filter_kwargs = seq_filter_kwargs_list[1]
# for i, similarity_kwargs in enumerate(similarity_kwargs_list):
#     cfg.similarity_kwargs = similarity_kwargs
#     cfg.bench_report_path = cfg.test_path / filter_names[1] / similarity_names[i]
#     cfg.frames_path = cfg.test_path / "frames.npz"
#     test = Test(cfg)
#     test.run()

In [11]:
rerank_k_list = [100, 250, 1000]
filter_names = ["base_seq_report", "nearfilter_seq_report", "farfilter_seq_report"]
cfg.seq_filter_kwargs = seq_filter_kwargs_list[2]
for k in rerank_k_list:
    cfg.rerank_k = k
    for i, similarity_kwargs in enumerate(similarity_kwargs_list):
        cfg.similarity_kwargs = similarity_kwargs
        cfg.bench_report_path = cfg.test_path / f"rerank_k_{k}" / filter_names[1] / similarity_names[i]
        cfg.frames_path = cfg.test_path / f"rerank_k_{k}" / "frames.npz"
        test = Test(cfg)
        test.run()

2026-05-07 00:13:55.262 | INFO     | mmpr.inference.index:generate:439 - Using existing meta.parquet
2026-05-07 00:13:55.262 | INFO     | mmpr.inference.index:generate:468 - Using existing descriptors.npy
2026-05-07 00:13:55.277 | INFO     | mmpr.inference.index:generate:486 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/256_graph_index
2026-05-07 00:13:55.318 | INFO     | mmpr.inference.index:generate:439 - Using existing meta.parquet
2026-05-07 00:13:55.318 | INFO     | mmpr.inference.index:generate:468 - Using existing descriptors.npy
2026-05-07 00:13:55.418 | INFO     | mmpr.inference.index:generate:486 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/megaloc_index


Running inference and saving frames to /home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/256xMegaloc/rerank_k_100/frames.npz
Using cached descriptors for query and rerank: loading from  /home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/256_graph_query_cache /home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/megaloc_query_cache
Descriptors loaded:  (21013, 256) (21013, 8448) getting results


retrieval: 21013it [00:29, 707.89it/s]


Results got:  21013


100%|██████████| 11/11 [05:01<00:00, 27.45s/it]
2026-05-07 00:19:29.437 | INFO     | mmpr.inference.index:generate:439 - Using existing meta.parquet
2026-05-07 00:19:29.438 | INFO     | mmpr.inference.index:generate:468 - Using existing descriptors.npy
2026-05-07 00:19:29.442 | INFO     | mmpr.inference.index:generate:486 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/256_graph_index
2026-05-07 00:19:29.481 | INFO     | mmpr.inference.index:generate:439 - Using existing meta.parquet
2026-05-07 00:19:29.481 | INFO     | mmpr.inference.index:generate:468 - Using existing descriptors.npy


...

Index1 search time mean: 0.0
Rerank index2 search time mean: 0.0


2026-05-07 01:20:56.938 | INFO     | mmpr.inference.index:generate:486 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/megaloc_index


Loading frames from /home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/256xMegaloc/rerank_k_1000/frames.npz


100%|██████████| 11/11 [09:36<00:00, 52.45s/it]

Index1 search time mean: 0.0
Rerank index2 search time mean: 0.0


In [10]:
k = 50
cfg.rerank_k = k
cfg.similarity_kwargs = similarity_kwargs_list[0]
cfg.bench_report_path = cfg.test_path / f"rerank_k_{k}" / filter_names[1] / similarity_names[0]
cfg.frames_path = cfg.test_path / f"rerank_k_{k}" / "frames.npz"
test = Test(cfg)
test.run()

2026-05-07 00:06:58.670 | INFO     | mmpr.inference.index:generate:439 - Using existing meta.parquet
2026-05-07 00:06:58.671 | INFO     | mmpr.inference.index:generate:468 - Using existing descriptors.npy
2026-05-07 00:06:58.674 | INFO     | mmpr.inference.index:generate:486 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/256_graph_index
2026-05-07 00:06:58.702 | INFO     | mmpr.inference.index:generate:439 - Using existing meta.parquet
2026-05-07 00:06:58.703 | INFO     | mmpr.inference.index:generate:468 - Using existing descriptors.npy
2026-05-07 00:06:59.124 | INFO     | mmpr.inference.index:generate:486 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/megaloc_index


Loading frames from /home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/256xMegaloc/rerank_k_50/frames.npz


100%|██████████| 11/11 [05:00<00:00, 27.35s/it]

Index1 search time mean: 0.0
Rerank index2 search time mean: 0.0


In [ ]:
similarity_names = ["room-sim", "pose-near-sim", "pose-far-sim"]
filter_names = ["base_seq_report", "nearfilter_seq_report", "farfilter_seq_report"]
cfg.seq_filter_kwargs = seq_filter_kwargs_list[2]
for i, similarity_kwargs in enumerate(similarity_kwargs_list):
    cfg.similarity_kwargs = similarity_kwargs
    cfg.bench_report_path = cfg.test_path / filter_names[2] / similarity_names[i]
    cfg.frames_path = cfg.test_path / "frames.npz"
    test = Test(cfg)
    test.run()

2026-05-06 17:52:58.599 | INFO     | mmpr.inference.index:generate:438 - Using existing meta.parquet
2026-05-06 17:52:58.600 | INFO     | mmpr.inference.index:generate:467 - Using existing descriptors.npy
2026-05-06 17:52:58.615 | INFO     | mmpr.inference.index:generate:485 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/256_graph_index
2026-05-06 17:52:58.670 | INFO     | mmpr.inference.index:generate:438 - Using existing meta.parquet
2026-05-06 17:52:58.671 | INFO     | mmpr.inference.index:generate:467 - Using existing descriptors.npy
2026-05-06 17:52:59.191 | INFO     | mmpr.inference.index:generate:485 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/megaloc_index


Loading frames from /home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/256xMegaloc/frames.npz


100%|██████████| 11/11 [04:58<00:00, 27.09s/it]
2026-05-06 17:57:58.071 | INFO     | mmpr.inference.index:generate:438 - Using existing meta.parquet
2026-05-06 17:57:58.072 | INFO     | mmpr.inference.index:generate:467 - Using existing descriptors.npy
2026-05-06 17:57:58.074 | INFO     | mmpr.inference.index:generate:485 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/256_graph_index
2026-05-06 17:57:58.104 | INFO     | mmpr.inference.index:generate:438 - Using existing meta.parquet
2026-05-06 17:57:58.104 | INFO     | mmpr.inference.index:generate:467 - Using existing descriptors.npy


Index1 search time mean: 0.0
Rerank index2 search time mean: 0.0


2026-05-06 17:57:58.631 | INFO     | mmpr.inference.index:generate:485 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/megaloc_index


Loading frames from /home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/256xMegaloc/frames.npz


 64%|██████▎   | 7/11 [06:10<03:40, 55.03s/it]

In [ ]:
cfg.similarity_kwargs = similarity_kwargs_list[1]
cfg.bench_report_path = cfg.test_path / filter_names[2] / similarity_names[1]
cfg.frames_path = cfg.test_path / "frames.npz"
test = Test(cfg)
test.run()

2026-05-06 18:11:44.841 | INFO     | mmpr.inference.index:generate:438 - Using existing meta.parquet
2026-05-06 18:11:44.842 | INFO     | mmpr.inference.index:generate:467 - Using existing descriptors.npy
2026-05-06 18:11:44.844 | INFO     | mmpr.inference.index:generate:485 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/256_graph_index
2026-05-06 18:11:44.877 | INFO     | mmpr.inference.index:generate:438 - Using existing meta.parquet
2026-05-06 18:11:44.878 | INFO     | mmpr.inference.index:generate:467 - Using existing descriptors.npy
2026-05-06 18:11:45.382 | INFO     | mmpr.inference.index:generate:485 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/megaloc_index


Loading frames from /home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/256xMegaloc/frames.npz


100%|██████████| 11/11 [09:58<00:00, 54.43s/it]

Index1 search time mean: 0.0
Rerank index2 search time mean: 0.0


In [ ]:
cfg.similarity_kwargs = similarity_kwargs_list[2]
cfg.bench_report_path = cfg.test_path / filter_names[2] / similarity_names[2]
cfg.frames_path = cfg.test_path / "frames.npz"
test = Test(cfg)
test.run()

2026-05-06 18:21:45.132 | INFO     | mmpr.inference.index:generate:438 - Using existing meta.parquet
2026-05-06 18:21:45.133 | INFO     | mmpr.inference.index:generate:467 - Using existing descriptors.npy
2026-05-06 18:21:45.135 | INFO     | mmpr.inference.index:generate:485 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/256_graph_index
2026-05-06 18:21:45.163 | INFO     | mmpr.inference.index:generate:438 - Using existing meta.parquet
2026-05-06 18:21:45.164 | INFO     | mmpr.inference.index:generate:467 - Using existing descriptors.npy
2026-05-06 18:21:45.222 | INFO     | mmpr.inference.index:generate:485 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/megaloc_index


Loading frames from /home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/256xMegaloc/frames.npz


100%|██████████| 11/11 [09:20<00:00, 50.98s/it]

Index1 search time mean: 0.0
Rerank index2 search time mean: 0.0


In [ ]:
test.run()

Loading frames from /home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/256xMegaloc/frames.npz


100%|██████████| 11/11 [09:37<00:00, 52.51s/it]

Index1 search time mean: 0.0
Rerank index2 search time mean: 0.0


In [ ]:
self.pipeline.index1_search_time_mean_s

NameError: name 'self' is not defined

In [ ]:
test.pipeline.index1_search_time_mean_s

0.000656297430468102

In [ ]:
test.pipeline.index2_search_time_mean_s

0.028109462976389252

In [ ]:
plot_metrics_from_parquet(
    "/home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/256xMegaloc/nearfilter_seq_report/pose-far-sim/summaryresults.parquet",
    "/home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/256xMegaloc/base_seq_report/pose-far-sim/summaryresults.parquet",
    metrics=("recall_at_1", "recall_at_5", "recall_at_10", "recall_at_25"))

{'recall_at_1': Figure({
     'data': [{'error_y': {'array': {'bdata': ('UA+BIHxhoj+jBJnytFuhPy5MlVjezZ' ... 'G67Jc/+DeN/IZfmj/EiOh6UzyZPw=='),
                                     'dtype': 'f8'}},
               'hovertemplate': 'w=%{x}<br>recall_at_1=%{y}<extra></extra>',
               'legendgroup': '',
               'line': {'color': '#636efa', 'dash': 'solid'},
               'marker': {'symbol': 'circle'},
               'mode': 'lines+markers',
               'name': '',
               'orientation': 'v',
               'showlegend': False,
               'type': 'scatter',
               'x': {'bdata': 'AQIDBQcKDxQZHiM=', 'dtype': 'i1'},
               'xaxis': 'x',
               'y': {'bdata': ('W82IV/OS6j/87PSYYSHsP8bXzQBG1u' ... 'kJC+4/yPP+fbLx7T/bv4wjstrtPw=='),
                     'dtype': 'f8'},
               'yaxis': 'y'},
              {'marker': {'color': 'red', 'size': 10},
               'mode': 'markers',
               'name': 'max',
               'showlegend

In [ ]:
plot_metrics_from_parquet(
    "/home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/256xMegaloc/nearfilter_seq_report/pose-far-sim/summaryresults.parquet",
    "/home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/256xMegaloc/base_seq_report/pose-far-sim/summaryresults.parquet",
    metrics=("recall_at_1", "recall_at_5", "recall_at_10", "recall_at_25"))

{'recall_at_1': Figure({
     'data': [{'error_y': {'array': {'bdata': ('ug8YavLgpj9UBrzqmMKnP7AcnNMfoa' ... 'z8M6c/nDfnGo9xqT8c1Gc0d5mnPw=='),
                                     'dtype': 'f8'}},
               'hovertemplate': 'w=%{x}<br>recall_at_1=%{y}<extra></extra>',
               'legendgroup': '',
               'line': {'color': '#636efa', 'dash': 'solid'},
               'marker': {'symbol': 'circle'},
               'mode': 'lines+markers',
               'name': '',
               'orientation': 'v',
               'showlegend': False,
               'type': 'scatter',
               'x': {'bdata': 'AQIDBQcKDxQZHiM=', 'dtype': 'i1'},
               'xaxis': 'x',
               'y': {'bdata': ('hzpME5AW5D+DuDvyQS/lP0dwAGyjsu' ... 'eZX+U/Iws2/Bo15T9DmuL4JzPlPw=='),
                     'dtype': 'f8'},
               'yaxis': 'y'},
              {'marker': {'color': 'red', 'size': 10},
               'mode': 'markers',
               'name': 'max',
               'showlegend

In [ ]:
plot_metrics_from_parquet(
    "/home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/256xMegaloc/nearfilter_seq_report/pose-near-sim/summaryresults.parquet",
    "/home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/256xMegaloc/base_seq_report/pose-near-sim/summaryresults.parquet",
    metrics=("recall_at_1", "recall_at_5", "recall_at_10", "recall_at_25"))

{'recall_at_1': Figure({
     'data': [{'error_y': {'array': {'bdata': ('GdFzl0HVpz/NFeRxMYSoPx4NtGDsU6' ... 'P8Fac/IeEAKdWTpz9kpF+HqDmnPw=='),
                                     'dtype': 'f8'}},
               'hovertemplate': 'w=%{x}<br>recall_at_1=%{y}<extra></extra>',
               'legendgroup': '',
               'line': {'color': '#636efa', 'dash': 'solid'},
               'marker': {'symbol': 'circle'},
               'mode': 'lines+markers',
               'name': '',
               'orientation': 'v',
               'showlegend': False,
               'type': 'scatter',
               'x': {'bdata': 'AQIDBQcKDxQZHiM=', 'dtype': 'i1'},
               'xaxis': 'x',
               'y': {'bdata': ('sGf9wGP01D8RW9nTh2fWP9BY7kTOqd' ... '/OVNM/OajQ9TO70j+H5j7BYn7SPw=='),
                     'dtype': 'f8'},
               'yaxis': 'y'},
              {'marker': {'color': 'red', 'size': 10},
               'mode': 'markers',
               'name': 'max',
               'showlegend

In [ ]:
plot_metrics_from_parquet(
    "/home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/256xMegaloc/farfilter_seq_report/room-sim/summaryresults.parquet",
    "/home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/256xMegaloc/nearfilter_seq_report/room-sim/summaryresults.parquet",
    metrics=("recall_at_1", "recall_at_5", "recall_at_10", "recall_at_25"))

{'recall_at_1': Figure({
     'data': [{'error_y': {'array': {'bdata': ('cUbi3p/Eoj/UO53XYuWgP/INdHIup5' ... 'xPTZ0/PwxUm4yonj+pJaQrp/WbPw=='),
                                     'dtype': 'f8'}},
               'hovertemplate': 'w=%{x}<br>recall_at_1=%{y}<extra></extra>',
               'legendgroup': '',
               'line': {'color': '#636efa', 'dash': 'solid'},
               'marker': {'symbol': 'circle'},
               'mode': 'lines+markers',
               'name': '',
               'orientation': 'v',
               'showlegend': False,
               'type': 'scatter',
               'x': {'bdata': 'AQIDBQcKDxQZHiM=', 'dtype': 'i1'},
               'xaxis': 'x',
               'y': {'bdata': ('iJi4kHWW6j/oLp4oklfsPxHVs7vNJe' ... 'ZuFe0/JaFBYc0O7T93LjKF30HtPw=='),
                     'dtype': 'f8'},
               'yaxis': 'y'},
              {'marker': {'color': 'red', 'size': 10},
               'mode': 'markers',
               'name': 'max',
               'showlegend

In [ ]:
plot_metrics_from_parquet(
    "/home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/256xMegaloc/farfilter_seq_report/pose-far-sim/summaryresults.parquet",
    "/home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/256xMegaloc/nearfilter_seq_report/pose-far-sim/summaryresults.parquet",
    metrics=("recall_at_1", "recall_at_5", "recall_at_10", "recall_at_25"))

{'recall_at_1': Figure({
     'data': [{'error_y': {'array': {'bdata': ('TYtABNdkpz858liCKbOnP3a/Xpwxf6' ... 'jbBKk/wmLbnVjLqT/qGvDYbE2oPw=='),
                                     'dtype': 'f8'}},
               'hovertemplate': 'w=%{x}<br>recall_at_1=%{y}<extra></extra>',
               'legendgroup': '',
               'line': {'color': '#636efa', 'dash': 'solid'},
               'marker': {'symbol': 'circle'},
               'mode': 'lines+markers',
               'name': '',
               'orientation': 'v',
               'showlegend': False,
               'type': 'scatter',
               'x': {'bdata': 'AQIDBQcKDxQZHiM=', 'dtype': 'i1'},
               'xaxis': 'x',
               'y': {'bdata': ('dNnrHcj+4z/i+hM4MVjlP8Aq3yp95u' ... '3pueQ/K3GbxRS75D8XLKk6reHkPw=='),
                     'dtype': 'f8'},
               'yaxis': 'y'},
              {'marker': {'color': 'red', 'size': 10},
               'mode': 'markers',
               'name': 'max',
               'showlegend

In [ ]:
plot_metrics_from_parquet(
    "/home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/256xMegaloc/farfilter_seq_report/pose-near-sim/summaryresults.parquet",
    "/home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/makarov/256xMegaloc/nearfilter_seq_report/pose-near-sim/summaryresults.parquet",
    metrics=("recall_at_1", "recall_at_5", "recall_at_10", "recall_at_25"))

{'recall_at_1': Figure({
     'data': [{'error_y': {'array': {'bdata': ('86+dfOQCqT9iDaSb7b2pPzV4+a9hEq' ... 'G76qc/ZkEtHyFupj+sqXd+mLanPw=='),
                                     'dtype': 'f8'}},
               'hovertemplate': 'w=%{x}<br>recall_at_1=%{y}<extra></extra>',
               'legendgroup': '',
               'line': {'color': '#636efa', 'dash': 'solid'},
               'marker': {'symbol': 'circle'},
               'mode': 'lines+markers',
               'name': '',
               'orientation': 'v',
               'showlegend': False,
               'type': 'scatter',
               'x': {'bdata': 'AQIDBQcKDxQZHiM=', 'dtype': 'i1'},
               'xaxis': 'x',
               'y': {'bdata': ('P6hWG0x11D/sEX3OXyjWP7fSsJazf9' ... 'ubq9I/rKj/q/Kt0j83Z0jljEfTPw=='),
                     'dtype': 'f8'},
               'yaxis': 'y'},
              {'marker': {'color': 'red', 'size': 10},
               'mode': 'markers',
               'name': 'max',
               'showlegend